# Chapter 18: Fine-Tuning Locally (Hugging Face + Unsloth)
## Topic 6: QLoRA Hyperparameters Explained

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topics 4 and 5 explained precisely what LoRA rank and 4-bit quantization mathematically do. This topic addresses the remaining configuration values Topic 3's real setup code specified without full explanation — `lora_alpha`, `learning_rate`, `epochs`, and `target_modules` — each a genuine, consequential choice requiring its own understanding, not an arbitrary value copied from an example without knowing what it controls.
- **Rank** (Topic 4) controls how much *capacity* the low-rank update matrices have. **Alpha** is a separate, distinct scaling factor applied to the LoRA update before it's added to the frozen base weights — the actual computation becomes `output = x @ (W + (alpha/rank) × B@A)`, meaning alpha and rank together control the *effective magnitude* of the LoRA update's influence on the model's behavior, not just its raw capacity. A common, empirically-motivated convention (which Topic 3's actual configuration uses) is setting alpha equal to rank, producing a scaling factor of exactly 1 — though this ratio itself is a genuine, tunable hyperparameter, not a fixed law.
- **Learning rate** and **epochs** control the training *process* itself, not the LoRA architecture — learning rate determines how large a step the optimizer takes when updating A and B's parameters based on each computed gradient, and epochs determine how many complete passes through Topic 2's real training dataset occur before training stops. These interact directly with Topic 4's rank choice: a LoRA configuration with more capacity (higher rank) may need different learning-rate tuning than a lower-rank configuration to train effectively without either learning too slowly or overshooting to instability.


### 2. Internal Working — Step by Step

**How each hyperparameter concretely affects training, step by step:**

1. **Alpha's scaling effect, precisely:** the LoRA update `B@A` gets multiplied by `alpha/rank` before being added to the frozen base weight — this means for a fixed rank, increasing alpha increases how strongly the learned update actually influences the model's output, while decreasing alpha diminishes this influence, even though the underlying matrices A and B themselves are being trained identically either way. This is a genuinely separate lever from rank itself: rank controls capacity (how complex an update can be represented), alpha controls magnitude (how strongly that update is actually applied).
2. **Learning rate's effect on training dynamics:** too high a learning rate risks the optimizer taking steps so large that training becomes unstable — loss oscillating or even increasing rather than decreasing; too low a learning rate risks training progressing so slowly that the model never reaches an acceptable result within a practical number of epochs — this is precisely the kind of hyperparameter-related failure mode Topic 4's own theory flagged as needing to be distinguished from an insufficient-rank problem.
3. **Epochs and the overfitting risk this introduces**: training for too many epochs on Topic 2's real, relatively small (431-example) hard-case dataset risks the model essentially memorizing the specific training examples rather than learning the generalizable behavioral pattern those examples were meant to demonstrate — this is precisely why Topic 2's genuine train/validation split matters so much: Topic 7's evaluation on the held-out validation portion is what actually reveals whether training has generalized or merely memorized, and monitoring validation performance (not just training loss) across epochs is the direct, practical signal for when to stop training.
4. **Target modules (Topic 3's specific list) determine where in the model's architecture the LoRA update is actually applied** — a narrower selection of target modules further reduces trainable parameters (extending Topic 4's efficiency math) at the potential cost of expressive reach across the model's full architecture; this project's actual choice (all seven attention and feed-forward projection matrices) provides broad coverage, a reasonable default absent specific evidence that a narrower selection would suffice for this project's specific fine-tuning goal.


### 3. How This Is Implemented in This Project

- Topic 3's actual, real configuration set `lora_alpha = 16` alongside `r = 16` — exactly matching the common alpha-equals-rank convention this topic describes, producing a clean scaling factor of 1 for this project's specific setup, a reasonable, well-established starting point rather than an arbitrary choice.
- This project's genuine train/validation split (Topic 2's real 344/87 division) is precisely the infrastructure needed to detect the overfitting risk this topic's theory describes — training loss alone, computed only on the training set, cannot reveal whether the model is genuinely learning the intended tone-consistency behavior or simply memorizing Topic 2's specific 344 training examples; Topic 7's evaluation against the separate, held-out 87 validation examples is what actually tests for genuine generalization.
- Given this project's genuinely small, targeted training dataset (431 total examples, following Topic 2's deliberate "narrow, focused hard-case dataset" design principle), a correspondingly conservative approach to epochs and learning rate is warranted — a small dataset reaches its risk of overfitting sooner (in fewer epochs) than a much larger dataset would, making careful, validation-monitored early stopping a particularly important practical consideration for this project's specific, real data scale.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Alpha and rank's combined ratio (alpha/rank) is what actually matters for the update's effective magnitude, not either value in isolation** — a common, easy misunderstanding is treating alpha as if it worked identically to rank (more capacity), when it's actually a distinct scaling lever; misdiagnosing an alpha-related instability as a rank-related capacity problem (or vice versa) leads to tuning the wrong hyperparameter.
- **A learning rate that's appropriate for one rank/alpha configuration may not transfer directly to a different configuration** — since alpha's scaling directly affects how strongly gradient updates translate into actual behavioral change, changing alpha (or rank) without reconsidering learning rate risks either an unstable or an overly-conservative training run, requiring re-validation rather than assuming a previously-good learning-rate value remains appropriate after other hyperparameters change.
- **Monitoring only training loss, without also tracking validation performance across epochs, is a common, serious mistake specifically given this project's small, 431-example dataset** — training loss can continue decreasing even as the model overfits to the specific training examples, while validation performance (checked using Topic 7's evaluation methodology) would reveal this overfitting directly, exactly the discipline this topic's theory requires.
- **Debugging an unstable or non-converging training run requires checking learning rate first, given it's the most direct, common cause of this specific symptom** — before suspecting a data problem (Topic 2) or an insufficient-rank problem (Topic 4), an obviously oscillating or diverging loss curve from very early in training most directly points toward a learning-rate issue, following the same layered-attribution discipline this notebook series has repeatedly required.
- **Monitoring:** tracking both training loss and validation-set performance (not just one or the other) throughout a training run, and specifically watching for the point where validation performance stops improving or starts degrading while training loss continues to decrease, is the direct, practical signal for the epoch count at which training should actually stop — a real, empirical determination rather than a fixed, assumed epoch count decided in advance.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Alpha-equals-rank convention (this project's actual choice) vs a different alpha/rank ratio:** the alpha-equals-rank convention is a reasonable, well-established starting point, producing a clean, easily-reasoned-about scaling factor of 1 — a different ratio is a genuine, tunable choice worth exploring empirically if this project's specific fine-tuning task shows signs of needing a stronger or weaker update magnitude than this default provides.
- **Conservative vs aggressive learning rate:** a more conservative (lower) learning rate reduces instability risk but requires more epochs (and correspondingly more risk of overfitting on this project's small dataset) to reach a comparable result; a more aggressive (higher) learning rate trains faster but risks instability — the right choice should be validated empirically by observing actual training loss behavior, not assumed from a generic, task-independent default value.
- **Fixed epoch count decided in advance vs validation-monitored early stopping:** a fixed epoch count is simpler to configure but risks either stopping training too early (before the model has genuinely learned the intended pattern) or too late (after overfitting has already begun) — validation-monitored early stopping (this topic's recommended practice, especially important given this project's small dataset) is more robust, at the cost of requiring the additional infrastructure to actually track validation performance during training, not just afterward.


### 6. Alternatives and When to Use Each

- **Alpha-equals-rank convention (this project's actual default):** a reasonable, well-established starting point for most LoRA fine-tuning tasks, worth adjusting only with specific empirical evidence a different ratio performs better for a specific task.
- **A conservative, lower learning rate with more epochs and validation-monitored early stopping:** the more robust default for a project with a small, targeted training dataset (this project's actual 431-example situation), directly mitigating the overfitting risk this topic's theory identifies as particularly relevant at this scale.
- **A more aggressive learning rate with fewer epochs:** worth considering specifically once a project has confirmed (through actual, empirical training runs) that this configuration reaches an acceptable result without the instability risk a naive, untested aggressive setting might introduce.


### 7. Common Mistakes and Production Failures

- Treating alpha as functionally equivalent to rank (both simply "more capacity"), rather than understanding alpha's distinct role as a scaling factor on the update's effective magnitude.
- Reusing a learning rate value from a different rank/alpha configuration without re-validating it for a new configuration, risking instability or overly slow training.
- Monitoring only training loss without also tracking validation-set performance, missing overfitting that training loss alone cannot reveal — a particularly serious risk given this project's small, 431-example dataset.
- Fixing an epoch count in advance without validation-monitored early stopping, risking either under-trained or overfit final results.
- Misdiagnosing an unstable, oscillating training loss curve as a data-quality or insufficient-rank problem rather than checking learning rate first, the most common and direct cause of this specific symptom.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the difference between what rank and alpha each control in a LoRA configuration?
  A: Rank controls the low-rank update's *capacity* — how complex a pattern the update matrices A and B can represent. Alpha is a separate scaling factor, applied as alpha/rank, controlling the update's *effective magnitude* — how strongly that learned update actually influences the model's output once applied to the frozen base weights. These are genuinely distinct levers, not interchangeable.

- Q: Why is validation-set monitoring particularly important for this project's specific fine-tuning task?
  A: This project's training dataset is genuinely small (431 total examples, per Topic 2's deliberate, targeted construction) — a small dataset reaches overfitting risk (the model memorizing specific training examples rather than learning the generalizable pattern) sooner than a much larger dataset would, making validation-set monitoring (checking whether held-out performance continues improving, rather than just training loss) a particularly important, not merely optional, practice at this scale.

**Intermediate**

- Q: Why might a learning rate that worked well for one LoRA configuration not transfer directly to a different rank/alpha configuration?
  A: Alpha's scaling directly affects how strongly a given gradient update translates into actual behavioral change in the model — changing alpha (or rank) changes this relationship, meaning a learning rate tuned for one configuration's specific update dynamics may produce instability or overly slow training under a different configuration, requiring re-validation rather than an assumption that a previously-good value remains appropriate.

- Q: How would you distinguish an unstable training run caused by too-high a learning rate from one caused by a data-quality problem?
  A: A learning-rate instability typically shows up as an oscillating or diverging loss curve from very early in training, often before the model has had a chance to process much of the training data meaningfully. A data-quality problem (Topic 2's concern) more often shows training loss decreasing reasonably, but the resulting model still performing poorly on held-out validation cases specifically — the timing and shape of the loss curve, checked first, is the direct, practical signal for distinguishing these two categories.

**Advanced**

- Q: Design a hyperparameter-tuning process for this project's actual fine-tuning task, addressing rank, alpha, learning rate, and epochs together.
  A: Start with Topic 3's actual, reasonable defaults (rank 16, alpha 16, a conservative initial learning rate) given this project's small dataset and narrow, targeted fine-tuning goal. Run a short initial training pass, monitoring both training loss and validation-set performance (Topic 2's real 87-example held-out split) together — if training loss decreases smoothly without oscillation, learning rate is likely appropriate; if validation performance continues improving alongside training loss for a reasonable number of epochs before plateauing or degrading, this identifies the appropriate stopping point empirically rather than by a fixed, assumed epoch count. Only if this initial run reveals training loss plateauing at an unsatisfactory level (Topic 4's own diagnostic) would rank become the next hyperparameter to reconsider, following the layered-attribution discipline this notebook series has established.

- Q: A teammate proposes training for many more epochs than initially planned, arguing "more training is always better." What's your concern given this project's specific dataset?
  A: Given this project's genuinely small, 431-example training dataset, training for many additional epochs beyond what validation performance actually supports risks the model overfitting — essentially memorizing Topic 2's specific training examples rather than learning the generalizable tone-consistency pattern those examples were meant to demonstrate. "More training" only helps up to the point where validation performance is still genuinely improving; beyond that point, additional epochs actively risk degrading the model's real, generalized performance, which only validation-set monitoring (not training loss alone) can reveal.

**Scenario-based**

- Q: During this project's actual fine-tuning run, training loss decreases smoothly and reaches a very low value, but validation-set performance (Topic 7's evaluation) shows the model still exhibits inconsistent tone on held-out cases. Walk through your diagnosis.
  A: A very low training loss with poor validation performance is a classic overfitting signature — the model has learned to reproduce Topic 2's specific 344 training examples very well, but this hasn't generalized to the held-out validation cases, meaning the model likely memorized surface patterns specific to the training examples rather than the genuine, generalizable tone-consistency behavior. This points toward reducing epochs (stopping training earlier, at the point where validation performance was still improving, before this overfitting set in), and potentially reconsidering whether Topic 2's training dataset itself needs to be larger or more diverse to support genuine generalization at this task's actual complexity.

**Follow-up questions to expect:**

- "How would you decide the right validation-performance metric to monitor during training, given this project's specific tone-consistency fine-tuning goal?"
- "What would you do if training loss and validation performance disagreed about which epoch produced the best model?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The overfitting risk this topic emphasizes, and the train/validation monitoring discipline used to detect it, are foundational machine learning concepts, not unique to LLM fine-tuning specifically** — the same principle (a model achieving low training error while failing to generalize to unseen data) applies to any supervised learning process, and this topic's specific application to LoRA fine-tuning is a direct instance of this much broader, foundational concern.
- **The alpha/rank scaling relationship is conceptually related to weight-initialization and learning-rate-scaling principles studied more generally in deep learning** — how strongly a parameter update actually affects a model's behavior, independent of the update's own internal complexity, is a recurring consideration across many different neural network training contexts, not unique to LoRA specifically.
- **This topic's hyperparameter discussion is the direct, practical culmination of Topics 4 and 5's mathematical foundations** — rank (Topic 4), quantization (Topic 5), and now alpha, learning rate, and epochs together form the complete, actionable configuration this project's real training process (Topic 3's setup code) actually requires, setting up Topic 7's evaluation of whatever specific configuration this project ultimately settles on.

### 10. Quick Revision Summary (for last-minute interview prep)

> Beyond rank (Topic 4) and quantization (Topic 5), several additional hyperparameters shape a QLoRA fine-tuning run concretely. Alpha is a distinct scaling factor (applied as alpha/rank) controlling the learned update's effective magnitude, separate from rank's control over the update's capacity — this project's actual configuration uses the common alpha-equals-rank convention (both set to 16), producing a clean scaling factor of 1. Learning rate and epochs control the training process itself: too high a learning rate risks unstable, oscillating training loss; too low risks impractically slow progress. Given this project's genuinely small, targeted training dataset (431 examples, Topic 2's real construction), overfitting is a particularly relevant risk — training loss can continue decreasing even as the model merely memorizes specific training examples rather than learning the generalizable behavioral pattern, making validation-set monitoring (not training loss alone) essential for determining when training should actually stop. Diagnosing a problematic training run requires the same layered-attribution discipline established throughout this notebook series: an early, oscillating loss curve points toward learning rate; a low training loss paired with poor validation performance points toward overfitting (too many epochs, or an insufficiently large/diverse dataset); a training loss that plateaus too high regardless of epochs points back toward Topic 4's rank as the likely limiting factor.


### Module 1: Real Alpha Scaling Effect on Update Magnitude

In [1]:

# ------------------------------------------------------------------
# MODULE 1: REAL alpha/rank scaling effect, computed with actual matrices
# ------------------------------------------------------------------

import numpy as np

np.random.seed(42)

HIDDEN_DIM_DEMO = 32
RANK = 8

# REAL LoRA matrices A and B, randomly initialized (B is typically
# zero-initialized in real LoRA, but we use small random values here
# to make the SCALING EFFECT visible and measurable)
A = np.random.randn(RANK, HIDDEN_DIM_DEMO) * 0.1
B = np.random.randn(HIDDEN_DIM_DEMO, RANK) * 0.1

raw_update = B @ A  # the raw, UNSCALED low-rank update

print("=" * 70)
print("MODULE 1: REAL ALPHA/RANK SCALING EFFECT, COMPUTED DIRECTLY")
print("=" * 70)
print(f"\nRaw update (B @ A) magnitude (Frobenius norm): {np.linalg.norm(raw_update):.4f}")

print(f"\n{'Alpha':<8} | {'Scaling Factor (alpha/rank)':>28} | {'Scaled Update Magnitude':>25}")
print("-" * 65)

for alpha in [4, 8, 16, 32, 64]:
    scaling_factor = alpha / RANK
    scaled_update = scaling_factor * raw_update
    scaled_magnitude = np.linalg.norm(scaled_update)
    print(f"{alpha:<8} | {scaling_factor:>28.2f} | {scaled_magnitude:>25.4f}")

print("\nCONFIRMED: the SAME underlying A and B matrices (identical training)")
print("produce a DIFFERENT effective update magnitude purely based on alpha --")
print("this is the REAL, computable distinction between rank (capacity, fixed")
print("here) and alpha (magnitude, varying) this topic's theory describes.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL ALPHA/RANK SCALING EFFECT, COMPUTED DIRECTLY

Raw update (B @ A) magnitude (Frobenius norm): 0.8892

Alpha    |  Scaling Factor (alpha/rank) |   Scaled Update Magnitude
-----------------------------------------------------------------
4        |                         0.50 |                    0.4446
8        |                         1.00 |                    0.8892
16       |                         2.00 |                    1.7784
32       |                         4.00 |                    3.5568
64       |                         8.00 |                    7.1136

CONFIRMED: the SAME underlying A and B matrices (identical training)
produce a DIFFERENT effective update magnitude purely based on alpha --
this is the REAL, computable distinction between rank (capacity, fixed
here) and alpha (magnitude, varying) this topic's theory describes.

Module 1 complete. Run Module 2 next.


### Module 2: Simulating Learning-Rate Effects on Training Stability

In [2]:

# ------------------------------------------------------------------
# MODULE 2: REAL simulation of learning-rate effects on training loss
# ------------------------------------------------------------------

def simulate_training_loss(learning_rate, num_steps=50):
    # a REAL, simplified simulation of gradient descent on a toy
    # quadratic loss surface -- genuinely exhibits the actual
    # instability/slow-convergence behavior different learning rates
    # produce, using real, computable optimization dynamics
    x = 10.0  # start far from the minimum (at x=0)
    losses = []
    for step in range(num_steps):
        loss = x ** 2
        losses.append(loss)
        gradient = 2 * x
        x = x - learning_rate * gradient
        if abs(x) > 1e6:  # diverged
            break
    return losses

print("=" * 70)
print("MODULE 2: LEARNING RATE EFFECTS ON TRAINING STABILITY (real simulation)")
print("=" * 70)

for lr, label in [(0.01, "too low"), (0.3, "appropriate"), (1.1, "too high")]:
    losses = simulate_training_loss(lr)
    final_loss = losses[-1]
    num_steps_completed = len(losses)
    diverged = num_steps_completed < 50
    print(f"\nLearning rate = {lr} ({label}):")
    print(f"  Loss after 5 steps:  {losses[4]:.4f}" if len(losses) > 4 else "  Diverged early")
    print(f"  Final loss ({num_steps_completed} steps): {final_loss:.4f}")
    print(f"  Diverged before completing all steps? {diverged}")

print("\nCONFIRMED: too LOW a learning rate leaves loss still far from the")
print("minimum after the same number of steps (slow progress); too HIGH a")
print("learning rate causes DIVERGENCE (loss increasing without bound) --")
print("both REAL, computable failure modes this topic's theory describes,")
print("using genuine (if simplified) optimization dynamics.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: LEARNING RATE EFFECTS ON TRAINING STABILITY (real simulation)

Learning rate = 0.01 (too low):
  Loss after 5 steps:  85.0763
  Final loss (50 steps): 13.8088
  Diverged before completing all steps? False

Learning rate = 0.3 (appropriate):
  Loss after 5 steps:  0.0655
  Final loss (50 steps): 0.0000
  Diverged before completing all steps? False

Learning rate = 1.1 (too high):
  Loss after 5 steps:  429.9817
  Final loss (50 steps): 5751248230.6955
  Diverged before completing all steps? False

CONFIRMED: too LOW a learning rate leaves loss still far from the
minimum after the same number of steps (slow progress); too HIGH a
learning rate causes DIVERGENCE (loss increasing without bound) --
both REAL, computable failure modes this topic's theory describes,
using genuine (if simplified) optimization dynamics.

Module 2 complete. Run Module 3 next.


### Module 3: Simulating Overfitting — Training Loss vs Validation Performance Divergence

In [3]:

# ------------------------------------------------------------------
# MODULE 3: REAL simulation of overfitting, training vs validation
# ------------------------------------------------------------------

def simulate_train_and_validation_curves(num_epochs=20):
    # REAL, computable simulation exhibiting genuine overfitting
    # dynamics -- training loss monotonically decreases, while
    # validation loss decreases then INCREASES past some epoch,
    # exactly the real pattern this topic's theory describes
    train_losses = []
    validation_losses = []
    for epoch in range(1, num_epochs + 1):
        train_loss = 2.0 * np.exp(-0.3 * epoch) + 0.05  # keeps decreasing
        # validation loss decreases initially, then increases past epoch ~8
        # (a real, computable U-shaped curve -- the actual overfitting signature)
        validation_loss = 2.0 * np.exp(-0.25 * epoch) + 0.02 * max(0, epoch - 8) ** 1.5 + 0.1
        train_losses.append(train_loss)
        validation_losses.append(validation_loss)
    return train_losses, validation_losses

train_losses, validation_losses = simulate_train_and_validation_curves()

print("=" * 70)
print("MODULE 3: OVERFITTING -- TRAINING vs VALIDATION LOSS (real simulation)")
print("=" * 70)
print(f"\n{'Epoch':<8} | {'Training Loss':>15} | {'Validation Loss':>17}")
print("-" * 45)
for epoch, (t_loss, v_loss) in enumerate(zip(train_losses, validation_losses), start=1):
    print(f"{epoch:<8} | {t_loss:>15.4f} | {v_loss:>17.4f}")

best_validation_epoch = validation_losses.index(min(validation_losses)) + 1
final_epoch = len(validation_losses)

print(f"\nBest validation performance reached at epoch: {best_validation_epoch}")
print(f"Training continued to epoch: {final_epoch}")
print(f"Training loss at final epoch: {train_losses[-1]:.4f} (still decreasing)")
print(f"Validation loss at final epoch: {validation_losses[-1]:.4f} (INCREASED from epoch {best_validation_epoch})")

if best_validation_epoch < final_epoch:
    print(f"\nCONFIRMED: training loss kept decreasing through epoch {final_epoch}, while")
    print(f"validation loss started INCREASING after epoch {best_validation_epoch} -- this IS the real,")
    print(f"computable overfitting signature this topic's theory describes. The")
    print(f"correct stopping point (validation-monitored early stopping) is epoch")
    print(f"{best_validation_epoch}, NOT the final epoch where training loss looks lowest.")

print("\nModule 3 complete. All Chapter 18 Topic 6 modules done.")
print()
print("=" * 70)
print("CHAPTER 18 TOPIC 6 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("Real matrix math confirmed: alpha scales the SAME underlying LoRA")
print("update to a DIFFERENT effective magnitude -- a genuinely separate")
print("lever from rank's capacity control.")
print()
print("Real, computable optimization dynamics demonstrated: too-low a")
print("learning rate leaves loss far from the minimum; too-high a learning")
print("rate causes genuine DIVERGENCE -- both real failure modes, not just")
print("described principles.")
print()
print("Real overfitting simulation: training loss monotonically decreased")
print("while validation loss showed the genuine U-shaped curve this topic's")
print("theory describes -- the correct stopping point is where VALIDATION")
print("performance peaks, not where training loss looks lowest.")
print()
print("This complete hyperparameter picture (rank, alpha, quantization,")
print("learning rate, epochs) is the direct configuration Topic 7's")
print("evaluation will assess against a prompted Claude baseline.")


MODULE 3: OVERFITTING -- TRAINING vs VALIDATION LOSS (real simulation)

Epoch    |   Training Loss |   Validation Loss
---------------------------------------------
1        |          1.5316 |            1.6576
2        |          1.1476 |            1.3131
3        |          0.8631 |            1.0447
4        |          0.6524 |            0.8358
5        |          0.4963 |            0.6730
6        |          0.3806 |            0.5463
7        |          0.2949 |            0.4475
8        |          0.2314 |            0.3707
9        |          0.1844 |            0.3308
10       |          0.1496 |            0.3207
11       |          0.1238 |            0.3318
12       |          0.1046 |            0.3596
13       |          0.0905 |            0.4012
14       |          0.0800 |            0.4543
15       |          0.0722 |            0.5174
16       |          0.0665 |            0.5892
17       |          0.0622 |            0.6685
18       |          0.0590 |        